# 牛津 Tutorial LLM 仿真 · MMM/MTA 增量测量 (v6.0)

## Persona Prompt (Oxford + HBS + Hattie)

> You are an Oxford tutorial fellow in **MMM/MTA 增量测量** (Marketing Mix Modeling + Multi-Touch Attribution + Incremental, Bayesian, DoWhy). Never give direct answers. Use Socratic questioning. Act as HBS devil's advocate. Reject vague claims like "MMM 比 MTA 好" or "DML 更稳健". End each turn with a probing question.

**Tutor 守则**:
1. **不直接给答案** (Never give direct answers) - 只追问、反问、给反例
2. **苏格拉底式追问** - 至少 5 类: 为什么 / 反例 / 若前提变 / 凭什么 / 如何
3. **HBS devil's advocate** - 拒绝模糊断言,要求用数学/数据支撑
4. **Hattie 四级反馈** - [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD] (避免 Self 级表扬)
5. **每轮以 probing question 结尾**

**主题范围**: MMM Adstock+Ridge / MTA 马尔可夫移除法 / NSW RCT 增量测量 / 合成控制 / DML 双重机器学习 / 贝叶斯 MMM / CUPED / 预算优化 + 增量验证


## Pre-Tutorial Task (强制 Retrieval)

> 牛津 tutorial 前置:学生必须先提交一段 essay/解题/方案。未提交者,tutor 拒绝开课。

**任务**: 用 300 字写一段 essay,主题:

> "给定 causaldata NSW 真实 RCT 数据 (445 样本, treat=培训干预, re78=1978 收入), 比较朴素均值差、合成控制、DML 三法估计 ATT 的优劣, 并说明在什么业务场景下该用哪法。要求: (1) 写出每法的数学假设 (2) 指出每法的失败场景 (3) 给出一个营销映射 (NSW -> 营销增量)"

**提交方式**: 把 essay 字符串赋给下方 `student_essay` 变量,tutorial loop 会基于内容追问。

**评分锚**: 0=未交, 1=只列方法名, 2=有数学假设, 3=有失败场景, 4=有营销映射, 5=四者齐全且批判性


In [ ]:
# Multi-turn Socratic Loop (静态 if/else 模拟, 不调 LLM API)
# 模拟 4 轮 Oxford tutor 追问, 基于学生 essay 内容分支

student_essay = """
NSW RCT 数据中, 朴素均值差 E[re78|treat=1]-E[re78|treat=0] 在全样本无偏, 因为随机化消除了混杂。
合成控制用加权对照组拟合反事实, 适合 RCT 不可行时。DML 用 ML 拟合混杂, 残差化后 OLS。
我觉得 DML 最稳健, 因为它能处理高维混杂。
"""  # 学生提交的 essay (示例, 可替换)

tutorial_log = []

def tutor_turn(round_num, essay_text):
    """静态 if/else 模拟 Oxford tutor 苏格拉底追问, 每轮一个 probing question"""
    round_num = int(round_num)
    if round_num == 1:
        # 追问 1: 为什么 (合成控制权重数学条件)
        return (
            "你说合成控制'用加权对照拟合反事实', 那权重 w 应该满足什么数学约束? "
            "若 pre-period 不匹配 (RMSE 高), 合成控制的 ATT 估计会朝哪个方向偏? "
            "凭什么相信加权对照能代表反事实?\n\n"
            ">>> 苏格拉底问 1 (为什么 + 凭什么): 合成控制权重的数学约束是什么?"
        )
    elif round_num == 2:
        # 追问 2: 反例 + 若前提变 (DML 双重去偏)
        return (
            "你说 DML '最稳健, 能处理高维混杂'。反例: 若 NSW 处理组与对照组在 re74/re75 上分布完全相同 "
            "(即无混杂), DML 与朴素均值差会给出相同的 theta 吗? 若不同, 偏差源在哪?\n\n"
            "DML 的'双重去偏'中, 两重分别是什么? 交叉拟合 (cross-fitting) 如何避免过拟合偏差? "
            "若用同一份数据训练 m(x) 和 g(x) 再残差化, 会发生什么?\n\n"
            ">>> 苏格拉底问 2 (反例 + 若前提变): 无混杂时 DML 与朴素均值差一致吗?"
        )
    elif round_num == 3:
        # 追问 3: 凭什么 (ATT vs ATE 业务场景)
        return (
            "你提到 ATT 但没说为何不用 ATE。凭什么用 ATT? 营销场景中, 哪个更贴近业务?\n\n"
            "反例: 某广告投放给'已会购买的用户' (incremental rate 2%) vs '创造新需求' (30%), "
            "ATE 与 ATT 哪个更能区分这两种场景? NSW 数据中, treat=1 是'自愿参加培训'的人, "
            "这本身已是一种 self-selection, 你如何用 ATT 的解释应对这种质疑?\n\n"
            ">>> 苏格拉底问 3 (凭什么 + 反例): ATT vs ATE 在 2% vs 30% 增量率场景中哪个更贴业务?"
        )
    elif round_num == 4:
        # 追问 4: 如何 (营销映射 + 增量验证)
        return (
            "你的营销映射太抽象。具体来说: NSW 的 treat -> 营销干预 (收到广告曝光), re78 -> 投放后消费, "
            "那么 re74/re75 作为协变量, 在营销场景中对应什么? CUPED (Microsoft 2013) 如何利用这些协变量缩减方差?\n\n"
            "如何用 Geo 实验 + CUPED 验证 MMM 预算优化结果的可信度? 若 MMM 说 TV 预算应增 30%, "
            "你如何设计一个 Geo 实验 + CUPED 来证伪/证实?\n\n"
            ">>> 苏格拉底问 4 (如何): 如何用 Geo+CUPED 验证 MMM 预算优化?"
        )
    else:
        return "Tutorial 结束。请阅读下方 Hattie 四级反馈, 并对照 student_model.json 修订盲点。"

# 跑 4 轮 Socratic loop
for r in range(1, 5):
    tutor_reply = tutor_turn(r, student_essay)
    tutorial_log.append({"round": r, "tutor": tutor_reply})
    print(f"\n===== Round {r} =====")
    print(tutor_reply)

print("\n===== Tutorial Loop End (4 rounds, 5+ Socratic questions) =====")


In [ ]:
# student_model.json 读写 (记录掌握度/盲点, Hattie 反馈依据)
import json, os

student_model = {
    "student_id": "S001",
    "unit": "elective-e2-marketing-analytics/day-3-mmm-mta-incremental",
    "mastery": {
        "MMM_Adstock_Ridge": 0.6,        # ILO2: R²>0.7 但 λ 匹配解释弱
        "MTA_Removal_Effect": 0.4,        # ILO3: 转移矩阵会建, Shapley 对比未掌握
        "Incremental_RCT": 0.7,           # ILO4: 朴素均值差懂, ATT/ATE 业务场景盲
        "Synthetic_Control": 0.3,         # ILO4: 权重约束说不出, pre-period RMSE 没报告
        "DML_CrossFitting": 0.3,          # ILO4: 双重去偏只背名词, 交叉拟合步骤跳过
        "Budget_Optimization": 0.5,       # ILO5: SLSQP 会调, KKT 不懂
        "CUPED_VarianceReduction": 0.2    # ILO5: 名词没听过
    },
    "blindspots": [
        "合成控制权重 w 的数学约束 (非负 + 和为 1 + pre-period 匹配)",
        "DML 交叉拟合 (cross-fitting) 的过拟合偏差规避机制",
        "ATT vs ATE 在增量率 2% vs 30% 业务场景的选择依据",
        "CUPED 用预处理协变量缩减 A/B 实验方差的数学推导",
        "KKT 条件 (拉格朗日乘子非负) 在预算优化中的检验"
    ],
    "essay_score": 2,  # 0-5 量表, 只有数学假设, 缺失败场景和营销映射
    "last_updated": "2026-07-26"
}

# 写入 student_model.json
model_path = os.path.join(os.getcwd(), "student_model.json")
with open(model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)
print(f"student_model.json 已写入: {model_path}")
print(f"掌握度均值: {sum(student_model['mastery'].values()) / len(student_model['mastery']):.2f}")
print(f"盲点数: {len(student_model['blindspots'])}")
print(f"essay 评分: {student_model['essay_score']}/5")


## Hattie 四级 Formative Feedback (基于 student_model.json)

> Hattie & Timperley (2007) 四级反馈模型。避免 Self 级表扬 (如"你做得很好"), 聚焦 Task / Process / Self-Reg / Feed-Forward。

### [TASK] 任务级反馈 (针对本次 essay 的具体内容)

- **问题**: 你写"合成控制用加权对照拟合反事实"但未写出权重 w 的三个数学约束: ① 非负 w_i >= 0 ② 和为 1 Σw_i = 1 ③ pre-period 匹配 RMSE 最小化
- **问题**: 你写"DML 最稳健"但未说明"双重"指什么 (答: 一重去 m(x)=E[T|X] 的偏, 二重去 g(x)=E[Y|X] 的偏)
- **修正**: 重写 essay, 补全每法的数学假设 (1 句话/法) + 失败场景 (1 句话/法)

### [PROCESS] 过程级反馈 (针对学习策略)

- **问题**: 你的 student_model.json 显示 DML_CrossFitting=0.3, 说明你跳过了交叉拟合 (cross-fitting) 步骤。这是 DML 区别于"残差回归"的关键
- **建议**: 回到 practice.md DRILL-03 faded 阶段, 手抄 2-fold cross-fitting 的 4 步流程 (拆样本 -> 训 m/g -> 残差化 -> OLS), 再做 independent 阶段
- **检测点**: 下次 tutorial 我会问"若用同一份数据训 m(x) 和 g(x) 再残差化, 偏差源在哪"

### [SELF-REG] 自我调节反馈 (针对元认知)

- **问题**: 你在 essay 中用"我觉得 DML 最稳健"--这是直觉而非论证。self-reg 弱
- **建议**: 下次写 essay 前, 先用 practice.md diagnostic D3 自测: 你能写出每法的失败场景吗? 若写不出, 说明你还没掌握, 不要下结论
- **元认知锚**: 每写一句断言, 自问"凭什么? 反例? 若前提变?" (Socratic 三问)

### [FEED-FORWARD] 前馈反馈 (指向下一步)

- **推荐复习单元** (基于 blindspots):
  1. schedule.json C3 (增量四法对比) + C4 (DML 算法) - 间隔复习 1 天后再测
  2. practice.md DRILL-03 worked (NSW 朴素均值差完整示范) - 手抄一遍
  3. notes.md § 关键回顾 4 (合成控制 + DML) - 重读后用自己的话复述
  4. 进阶: schedule.json C5 (贝叶斯 MMM) + C6 (CUPED + 预算优化) - 为 ILO5 做准备
- **下次 tutorial 主题**: 营销映射 + CUPED 方差缩减 + Geo 实验设计
- **mastery 路径**: 当前均值 0.43, 目标 0.80, 预计 3 次 tutorial (每周 1 次)


## 限频与 Exit Artifact

### 限频 (防依赖)

- **每单元 1 次/天**: tutorial 每天 1 次, 防止学生用 LLM 替代思考
- **强制间隔**: 两次 tutorial 之间至少 24 小时, 期间必须完成 practice.md 至少 1 个 drill
- **进度门控**: 上次 tutorial 的 [FEED-FORWARD] 复习任务未完成, 拒绝开新课
- **限频原因**: Hattie 研究表明, 过频反馈会削弱学生自我调节能力 (Self-Reg 退化)

### Exit Artifact (本次 tutorial 产出)

> 提交后才能结束本次 tutorial。

**2-3 盲点** (从 student_model.json 选):
1. 合成控制权重 w 的数学约束 (非负 + 和为 1 + pre-period 匹配) - 当前 mastery 0.3
2. DML 交叉拟合 (cross-fitting) 的过拟合偏差规避机制 - 当前 mastery 0.3
3. CUPED 用预处理协变量缩减 A/B 实验方差的数学推导 - 当前 mastery 0.2

**推荐复习单元**:
- practice.md DRILL-03 (NSW 增量测量四法对比) - 从 worked 重做
- schedule.json C3 + C4 - FSRS-6 间隔复习 [1, 3, 8, 21, 60, 180]
- notes.md § 关键回顾 4 (合成控制 + DML) + § 2026 前沿 (CUPED)
- 进阶: alignment.md ILO4 + ILO5 的 TLA 链路

**自检**: 完成 exit artifact 后, 跑 `python3 /tmp/verify_v6_unit.py` 确认 mastery_threshold 进度。下次 tutorial 前, tutorial.ipynb cell4 的 student_model.json 应显示 DML_CrossFitting >= 0.6。

---

*本 tutorial.ipynb 基于 Oxford tutorial + HBS case method + Hattie 四级反馈。Socratic loop 用静态 if/else 模拟, 不调 LLM API。限频机制防 LLM 依赖。*
